## 调用工具的三种核心方式

* 手动绑定（`bing_tools`） \
* 高级智能体（`crate_agent`） \
* 底层图执行（`LangGraph`+`ToolNode`）

In [2]:
# 示例1：最小示（需记下来）
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
# 获取llm模型
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
model = ChatOpenAI(
    model='MiniMax-M2.5',
    # model='MiniMax-M2.5',
    temperature=0.6
)


# @tool
def get_weather(city: str) -> str:
    """查询不同城市的天气情况"""
    # print('+++++')
    return f"{city}是下雪"


agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt='你是一个天气助手'
)
result = agent.invoke({
    "messages": [{"role": "user", "content": "帮我查询北京的天气"}]
})
print(result['messages'][-1].content)


根据查询结果，北京目前是**下雪**天气 ❄️。天气比较寒冷，出门记得注意保暖和防滑哦！有什么其他需要帮助的吗？


In [6]:
# 多伦对话，嵌入记忆
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.tools import TavilySearchResults
import os
import dotenv

dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
# 获取llm模型
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
model = ChatOpenAI(
    model='MiniMax-M2.5',
    # model='MiniMax-M2.5',
    temperature=0.6
)

# 使用工具
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
# 获取前三条网页记录
search = TavilySearchResults(max_results=3)
print( search)
tools = [search]

memory = MemorySaver()

agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=memory,  #关键！让 agent 记住历史
    system_prompt='你是一个联网的AI助手'
)
config = {"configurable": {"thread_id": "conversation-001"}}

'''
config=config 的用处就是让 Agent 知道你要操作哪个会话，
从而利用 checkpointer 实现跨轮次的对话记忆。没有它，每次调用都是全新的开始。
'''
# 第一轮
agent.invoke({
    "messages": [{"role": "user", "content": "今日北京的天气怎么样？"}]
}, config=config)

# 第二轮
res = agent.invoke({
    "messages": [{"role": "user", "content": "上海呢？"}]
}, config=config)

# print(res["messages"][-1].content)



max_results=3 api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'))


In [45]:
# 流式输出
for chunk in agent.stream(
        {"messages": [{"role": "user", "content": "南京呢"}]},
        config=config,
        # 想调试 Agent 行为（看它用了哪些工具、中间结果对不对）→ 用 updates
        # 想给用户看流畅的打字效果 → 用 messages
        # 既想看过程又想实时输出 → 可以同时用两种模式（传 stream_mode=["updates", "messages"]）
        stream_mode="messages"  # 或 "messages" 看具体需求
):
    # print(chunk)
    print(chunk[0].content, end="", flush=True)



[{"title": "南京 - 中国气象局-天气预报-城市预报", "url": "https://weather.cma.cn/web/weather/58238.html", "content": "|  |  |  |  |  |  |  |  |  |\n ---  ---  ---  --- \n| 时间 | 08:00 | 11:00 | 14:00 | 17:00 | 20:00 | 23:00 | 02:00 | 05:00 |\n| 天气 |\n| 气温 | 12.5℃ | 14.8℃ | 15.1℃ | 14.8℃ | 14.6℃ | 14.5℃ | 14.5℃ | 14.4℃ |\n| 降水 | 无降水 | 4.9mm | 无降水 | 4.9mm | 0.1mm | 24.6mm | 0.1mm | 24.6mm |\n| 风速 | 7.9m/s | 7.4m/s | 7.9m/s | 7.6m/s | 6.4m/s | 3.2m/s | 1.9m/s | 1.3m/s |\n| 风向 | 东南风 | 东南风 | 东南风 | 东南风 | 东南风 | 东南风 | 东南风 | 东南风 |\n| 气压 | 1011.6hPa | 1009.7hPa | 1007.7hPa | 1006.9hPa | 1006.2hPa | 1004.7hPa | 1003.2hPa | 1003.6hPa |\n| 湿度 | 70.6% | 64.7% | 64.6% | 64.5% | 76.5% | 93.3% | 97.5% | 92.4% |\n| 云量 | 80% | 70% | 61.6% | 78.2% | 94.8% | 95.7% | 96.6% | 97.7% | [...] |  |  |  |  |  |  |  |  |  |\n ---  ---  ---  --- \n| 时间 | 08:00 | 11:00 | 14:00 | 17:00 | 20:00 | 23:00 | 02:00 | 05:00 |\n| 天气 |\n| 气温 | 12.6℃ | 18.7℃ | 19.3℃ | 18.5℃ | 15.5℃ | 12.3℃ | 12.1℃ | 11.5℃ |\n| 降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水

In [51]:
# 结构化输出
from pydantic import BaseModel


'''
    需要注意模型输出的信息中有这些字段才行
'''
# 结构化模型
class WeatherReport(BaseModel):
    city:str
    temperature:float
    condition:str

agent=create_agent(
    model=model,
    tools=tools,
    response_format=WeatherReport, # 强制结构化返回
    system_prompt='你是一个联网的AI助手'
)
result=agent.invoke({"messages":[{"role":"user","content":"安徽呢"}]})
weather = result.get("structured_response")  # WeatherReport 对象
print(weather.city, weather.temperature)


{'messages': [HumanMessage(content='安徽呢', additional_kwargs={}, response_metadata={}, id='32e596f2-2e08-46f7-bbac-282f2a033fe2'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 281, 'total_tokens': 375, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 66, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 130}}, 'model_provider': 'openai', 'model_name': 'MiniMax-M2.5', 'system_fingerprint': None, 'id': 'chatcmpl-55a78bdf-18a7-9804-adff-13c73a4cdc17', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d62bf-87b4-7842-86db-b1027ac65090-0', tool_calls=[{'name': 'WeatherReport', 'args': {'city': '安徽'}, 'id': 'call_b2f41207dd064022a13729ea', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 281, 'output_tokens': 94, 'total_tokens': 375, 'input_token_deta

In [ ]:
# 示例2:旧版本

from langchain_core.tools import Tool
import os
import dotenv
from langchain_community.tools import TavilySearchResults, BingSearchResults
from langchain_openai import ChatOpenAI

dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')

# 获取Tavily搜索工具的实例
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
search = TavilySearchResults(max_results=3)

# 获取一个搜索的工具
search_tool = Tool(
    name="Search",
    func=search.run,
    description="用于搜索互联网上的信息"
)

# 获取llm模型
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
llm = ChatOpenAI(
    model='glm-5',
    temperature=0.6
)

# 通过create_agent调用invoke()，并得到响应


# 处理响应数据





In [9]:
# 示例3:使用create_agent
from langchain.agents import create_agent

import os
import dotenv
from langchain_community.tools import TavilySearchResults

from langchain_openai import ChatOpenAI

dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
# 获取llm模型
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
model = ChatOpenAI(
    model='glm-5',
    temperature=0.6
)
# 获取Tavily搜索工具的实例
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

# 获取前三条网页记录
search = TavilySearchResults(max_results=3)

tools = [search]
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="你是一个搜索AI助手"
)
# 3. 运行 Agent
result = agent.invoke({
    "messages": [
        {"role": "user", "content": "查询一下今日的实时热点"}
    ]
})

# print(result)
print(result["messages"][-1].content)


根据今日实时热点搜索结果，以下是当前最受关注的热点事件：

## 🏆 体育赛事热点

**乒乓球世界杯**
- **王楚钦首夺世界杯男单冠军**（热度最高）
- **孙颖莎实现世界杯单打三连冠**
- 王曼昱与孙颖莎会师女单决赛

**斯诺克赛事**
- 赵心童夺得2026斯诺克巡回锦标赛冠军

## 🌍 国际局势

**伊朗-美国冲突**
- 美军战机被伊朗击落，美军展开营救行动
- 伊朗称美军营救飞行员任务失败，多架飞机被击落
- 霍尔木兹海峡局势紧张

## 📱 社会热点

**娱乐圈**
- 郭艾伦遭诈骗千万元
- 周杰伦3小时演唱会实唱不足90分钟引争议
- 黄晓明因交通违规接受处罚

**财经科技**
- 25岁天才少女拿下超14亿融资
- 微信宣布某项功能将全面停运
- 六大行房贷减7100亿

**民生话题**
- 佛山莫氏鸡煲店火到限流，老板喊话想关门
- 清明节祭祖相关话题（祭祖金元宝卖断货）
- 警方紧急提醒苹果手机用户

## 🔥 网络热议

- 借1200元30年后还2万被拒
- 妈妈不知儿子去世跟AI儿子聊了1年
- 徒步大军堵死南太行

这些热点涵盖了体育、国际政治、社会民生、娱乐等多个领域，反映了当前公众最关注的话题。


In [19]:
# 多工具之间调用
from langchain.agents import create_agent
from langchain_core.tools import tool
import os
import dotenv
from langchain_community.tools import TavilySearchResults

from langchain_openai import ChatOpenAI

dotenv.load_dotenv(dotenv_path='../chapter02-model IO/.env')
# 获取llm模型
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY2")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL2")
model = ChatOpenAI(
    model='MiniMax-M2.5',
    temperature=0.6
)
# 获取Tavily搜索工具的实例
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")


# 分析工具
@tool
def analyze_stock_change(company_name: str, price_data: dict) -> str:
    """分析公司股价从年初至今的涨跌百分比。

    Args:
        company_name: 公司名称
        price_data: 包含 'current_price' 和 'start_price' 的字典
    """
    current_price = price_data['current_price']
    start_price = price_data['start_price']

    print(f"start_price==={start_price}")
    change_percent = ((current_price - start_price) / start_price) * 100
    return f"{company_name}股价从年初至今上涨/下跌了{change_percent:.2f}%"


# 获取前三条网页记录
search = TavilySearchResults(max_results=3)

tools = [search, analyze_stock_change]
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt='你是一个网页搜索助手。'
)
# 执行任务
res = agent.invoke({
    "messages": [{"role": "user", "content": "当前是2026年4月6日，宁德时代（300750）当前股价是多少？今年上涨了百分之几？"}]
})
print(res['messages'][-1].content)



start_price===367.26
根据我搜索到的数据，为您提供宁德时代（300750）2026年4月6日的股价信息：

## 2026年4月6日宁德时代股价

**当前股价约为：389.16元**（根据4月3日最新数据）

由于4月6日是周一，我找不到该日的精确交易数据，但从4月2日（401.17元）和4月3日（389.16元）的数据来看，股价在390-401元区间波动。

## 年初至今涨幅

**今年上涨约：5.96%**

计算依据：
- 2025年12月31日收盘价：367.26元
- 当前股价：389.16元
- 涨幅：(389.16 - 367.26) ÷ 367.26 × 100% = **5.96%**

## 近期股价走势

从数据来看，宁德时代进入2026年后股价有所波动：
- 1月初：约350-370元
- 1月底：约340-350元  
- 2月：约345-370元
- 3月：约340-420元（波动较大）
- 4月初：约390-405元

值得注意的是，宁德时代在2025年10月9日曾达到历史最高价424.36元，去年全年涨幅为61.42%。

以上数据仅供参考，实际交易价格请以交易所实时数据为准。


你好，李雷！很高兴认识你。

不过作为AI助手，我无法实时获取当前的日期信息（因为我没有接入实时日历系统）。你可以通过以下方式查看今天的日期：

1. **查看你的手机或电脑** - 屏幕右上角或任务栏通常会显示当前日期
2. **询问语音助手** - 比如 Siri、小爱同学等
3. **搜索引擎** - 搜索"今天几号"或"当前日期"

如果你想知道特定信息（比如某个日期是星期几，或者计算日期差），只要告诉我具体的日期，我很乐意帮你计算！

有什么其他我可以帮你的吗？
